## Model Training

#### 1.1 Import Data and Required Packages
##### Importing Pandas, Numpy, Matplotlib, Seaborn and Warings Library.

In [2]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

#### Import the CSV Data as Pandas DataFrame

In [4]:
df = pd.read_csv("./Data/after_preprocess.csv")

#### Show Top 5 Records

In [5]:
df.head()

,NAME,DIV,DEP,LOC,LEAVE DATE,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-01-31,CS,1.0,1,2020,4,2020-01-11,20.0,2020-02-01,1.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-01,CS,0.5,2,2020,5,2020-01-31,1.0,2020-02-15,14.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-15,AL,0.5,2,2020,5,2020-02-01,14.0,2020-02-18,3.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-02-18,CS,1.0,2,2020,1,2020-02-15,3.0,2020-03-10,21.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,2020-03-10,CS,1.0,3,2020,1,2020-02-18,21.0,2020-06-11,93.0,6


In [7]:
df = df.drop(columns=['LEAVE DATE', 'PREV LEAVE DATE', 'NEXT LEAVE DATE'])
df

,NAME,DIV,DEP,LOC,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,DAYS SINCE LAST LEAVE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,1,2020,4,20.0,1.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,0.5,2,2020,5,1.0,14.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,AL,0.5,2,2020,5,14.0,3.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,2,2020,1,3.0,21.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,3,2020,1,21.0,93.0,6
...,...,...,...,...,...,...,...,...,...,...,...,...
232796,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,SK,1.0,7,2024,1,20.0,60.0,3
232797,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,DL,0.5,9,2024,5,60.0,11.0,4
232798,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,CS,1.0,10,2024,2,11.0,5.0,5
232799,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,SK,1.0,10,2024,0,5.0,1.0,6


#### Preparing X and Y variables

In [8]:
X = df.drop(columns=['DAYS UNTIL NEXT LEAVE'],axis=1)

In [9]:
X.head()

,NAME,DIV,DEP,LOC,TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,DAYS SINCE LAST LEAVE,CUMULATIVE LEAVE COUNT
0,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,1,2020,4,20.0,2
1,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,0.5,2,2020,5,1.0,3
2,A.A.A.R.DE SILVA,DHA,SAB,SEU,AL,0.5,2,2020,5,14.0,4
3,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,2,2020,1,3.0,5
4,A.A.A.R.DE SILVA,DHA,SAB,SEU,CS,1.0,3,2020,1,21.0,6


In [12]:
y = df['DAYS UNTIL NEXT LEAVE']
y

0          1.0
1         14.0
2          3.0
3         21.0
4         93.0
          ... 
232796    60.0
232797    11.0
232798     5.0
232799     1.0
232800    37.0
Name: DAYS UNTIL NEXT LEAVE, Length: 232801, dtype: float64

#### Identifying categorical and numerical features

In [13]:
# define numerical & categorical columns
numeric_features = [feature for feature in df.columns if df[feature].dtype != 'O']
categorical_features = [feature for feature in df.columns if df[feature].dtype == 'O']

# print columns
print('We have {} numerical features : {}'.format(len(numeric_features), numeric_features))
print('\nWe have {} categorical features : {}'.format(len(categorical_features), categorical_features))

We have 7 numerical features : ['NO DAYS', 'LEAVE MONTH', 'LEAVE YEAR', 'LEAVE DAY OF WEEK', 'DAYS SINCE LAST LEAVE', 'DAYS UNTIL NEXT LEAVE', 'CUMULATIVE LEAVE COUNT']

We have 5 categorical features : ['NAME', 'DIV', 'DEP', 'LOC', 'TYPE']


In [14]:
print("Categories in 'NAME' variable:     ",end=" " )
print(df['NAME'].unique())

print("Categories in 'DIV' variable:  ",end=" ")
print(df['DIV'].unique())

print("Categories in 'DEP' variable:",end=" " )
print(df['DEP'].unique())

print("Categories in 'LOC' variable:     ",end=" " )
print(df['LOC'].unique())

print("Categories in 'TYPE' variable:     ",end=" " )
print(df['TYPE'].unique())

Categories in 'NAME' variable:      ['A.A.A.R.DE SILVA' 'A.A.C.P.AMARASINGHE' 'A.A.D.A.R.AMARASINGHE' ...
 'Y.P.L.R.K. DISSANAYAKE' 'Y.R.A.P.M. RAJAPAKSE' 'Y.S.D.DHARMATHILAKA']
Categories in 'DIV' variable:   ['DHA' 'DPR' 'DYA' 'DBU' 'DMD' 'DMP' 'DFI' 'DPH' 'DPJ' 'DRD' 'DCH' 'DQA'
 'DPE']
Categories in 'DEP' variable: ['SAB' 'SSO' 'SHC' 'SDE' 'SMT' 'SDM' 'SHE' 'SRC' 'SHD' 'SHT' 'SWE' 'SDS'
 'SMO' 'SEA' 'SQC' 'SSC' 'SMG' 'SMP' 'SAC' 'SIR' 'SEM' 'SIT' 'SSR' 'SSM'
 'SYD' 'SMA' 'SLE' 'SFI' 'SIN' 'SPJ' 'SPR' 'SAA' 'SBU' 'SCH' 'SHA' 'SMN'
 'SES' 'SNE' 'SQA' 'SMD' 'SRE']
Categories in 'LOC' variable:      ['SEU' 'COM' 'SWC' 'FIS' 'PPH' 'FIE' 'DOS' 'GAC' 'DFS' 'TRC' 'WTS' 'FIU'
 'PP4' 'SWR' 'MAT' 'BLP' 'KIT' 'DEO' 'MAO' 'MEW' 'SEE' 'ELW' 'ENW' 'MEC'
 'AUW' 'PFS' 'ALW' 'QCO' 'ADU' 'PPM' 'TRA' 'CAL' 'SEC' 'MPO' 'NTS' 'PEO'
 'PAY' 'BLU' 'PE1' 'SCA' 'EST' 'DMO' 'WHA' 'ITO' 'STD' 'SVG' 'WEP' 'STC'
 'FTS' 'MAS' 'KBL' 'WEL' 'SRP' 'YDU' 'SHB' 'WSO' 'LOP' 'MAR' 'PFH' 'PE2'
 'CAR' 'IMP' 'SAF' 'SHR' 'LE